In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras import models
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download and load the training data
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Download and load the test data
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

In [ ]:
# Define the CNN architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # First convolutional layer (1 input channel, 32 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional layer (32 input channels, 64 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # MNIST images are 28x28. After two 2x2 pooling layers, size becomes 28/2 -> 14/2 -> 7x7
        # So, input features to the linear layer = 64 channels * 7 * 7 image size
        self.fc1 = nn.Linear(64 * 7 * 7, 128) 
        self.fc2 = nn.Linear(128, 10) # Output layer (10 classes for MNIST digits)

    def forward(self, x):
        # Apply first conv layer, then ReLU, then pooling
        x = self.pool1(F.relu(self.conv1(x)))
        # Apply second conv layer, then ReLU, then pooling
        x = self.pool2(F.relu(self.conv2(x)))
        
        # Flatten the output from conv layers before passing to linear layers
        x = x.view(-1, 64 * 7 * 7) # -1 infers batch size
        
        # Apply first linear layer, then ReLU
        x = F.relu(self.fc1(x))
        # Apply second linear layer (output layer)
        x = self.fc2(x)
        return x

# Instantiate the model
model = Net()

# Print model structure (optional)
print(model)

In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train the model
num_epochs = 5
train_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        images, labels = data
        
        optimizer.zero_grad() # Zero the parameter gradients
        
        outputs = model(images) # Forward pass
        loss = criterion(outputs, labels) # Calculate loss
        loss.backward() # Backward pass
        optimizer.step() # Optimize
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(trainloader)
    train_losses.append(epoch_loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    
    # Validation for this epoch
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad(): # Disable gradient calculations
        for data in testloader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_acc = 100 * correct / total
    val_accuracies.append(epoch_acc)
    print(f'Accuracy of the network on the test images after epoch {epoch + 1}: {epoch_acc:.2f} %')

print('Finished Training')

In [ ]:
# Final evaluation of the model
model.eval() # Set model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total
print(f'Final accuracy of the network on the 10000 test images: {final_accuracy:.2f} %')

In [ ]:
import matplotlib.pyplot as plt

# Assuming train_losses and val_accuracies are populated from the training loop
# and num_epochs variable is available from the training cell, or derive from list length.
num_epochs_actual = range(1, len(train_losses) + 1)

# Plot 1: Training Loss vs. Epochs
plt.figure()
plt.plot(num_epochs_actual, train_losses, label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss over Epochs')
plt.show()

# Plot 2: Validation Accuracy vs. Epochs
plt.figure()
plt.plot(num_epochs_actual, val_accuracies, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('Validation Accuracy over Epochs')
plt.show()